## CAMP2Ex Sonde Plotting Code (Questions?  Email brodenkirch@wisc.edu)

#### Given a directory that contains directories for each CAMP2Ex research flight (that each contain QCed dropsonde/radiosonde CSV files, see _CAMP2Ex_dropsonde.ipynb_ and _CAMP2Ex_radiosonde.ipynb_), this code:

Calculates mean-layer metrics (e.g., RH, wind shear, CAPE, etc.) for each dropsonde/radiosonde using SHARPpy. The metrics for all sondes are then outputted to a CSV file (`Sonde_Metric_Calculations_CAMP2Ex_NEW.csv`) in the directory of this script **(NOTE:  some of these metrics purposefully are blank values, mostly because these blank metrics were meant to be manually inputted and/or were not useful/relevant anymore for the author)**.  The mid layer is defined as extending from the PBL top up to the freezing level.  The upper layer is defined as extending from the freezing level up to the _lowmax_hght_ **(you probably will want to change this!)**.  The deep layer is defined as extending from the near-surface (the given sonde's lowest available height level) up to the _lowmax_hght_.

Metrics that are calculated (in order of appearance on `Sonde_Metric_Calculations_CAMP2Ex_NEW.csv`) are:
* Sfc Pressure [mb]
* Sfc Height [m]
* PBL Top [mb] 
* PBL Top Height [m] 
* Freezing Level [mb] 
* Freezing Level Height [m]
* Upper Level Cap [mb] 
* Upper Level Cap Height [m]
* M10 Celsius Level [mb] 
* M20 Celsius Level [mb]
* M30 Celsius Level [mb]
* PBL RH [%]
* Mid Layer RH [%] 
* Upper Layer RH [%]
* Deep Layer RH [%]
* Below FZL MUCAPE [J/kg] 
* Above FZL MUCAPE [J/kg] 
* Deep Layer MUCAPE [J/kg] 
* MU CIN [J/kg]
* MU LCL [mb]
* MU EL [mb] 
* MU Max LI [degC] 
* MU Max LI Level [mb] 
* MU Max LI Layer **(always blank)**
* Below FZL MLCAPE [J/kg]
* Above FZL MLCAPE [J/kg]	
* Deep Layer MLCAPE [J/kg]
* ML CIN [J/kg]
* ML LCL [mb]
* ML EL [mb] 
* ML Max LI [degC]
* ML Max LI Level [mb] 
* ML Max LI Layer **(always blank)**
* SHARPpy Direct Method PBL Speed Shear [kts]	
* SHARPpy Direct Method PBL Directional Shear [deg] 
* SHARPpy Direct Method Mid Layer Speed Shear [kts] 
* SHARPpy Direct Method Mid Layer Directional Shear [deg] 
* SHARPpy Direct Method Upper Layer Speed Shear [kts]	
* SHARPpy Direct Method Upper Layer Directional Shear [deg]
* SHARPpy Direct Method Deep Layer Speed Shear [kts] 
* SHARPpy Direct Method Deep Layer Directional Shear [deg] 
* 10mb Interval Theta Gradient PBL Top [mb] **(always blank)**
* 10mb Interval Theta Gradient PBL RH [%] **(always blank)**
* 10mb Interval Theta Gradient Mid Layer RH [%] **(always blank)**
* Max Profile Height [m]
* 500m Bottom Cap Profile Height [m] 
* 500m Bottom Cap Deep Layer Speed Shear [kts] 
* 500m Bottom Cap Deep Layer Directional Shear [deg]

### Hope this helps!  Feel free to edit the code to fit your needs.  The variables you will definitely want to change right away are _day_folder_ and the _lowmax_hght_ parameter (cell #3).  Once you change these, everything should run properly as is.  If it doesn't, let the author know (brodenkirch@wisc.edu).

In [ ]:
import os
import sys
#import xarray as xr
import pandas as pd
import numpy as np
#from datetime import datetime

#import sharppy
import sharppy.sharptab.profile as profile
#import sharppy.sharptab.interp as interp
import sharppy.sharptab.winds as winds
import sharppy.sharptab.utils as utils
import sharppy.sharptab.params as params
#import sharppy.sharptab.thermo as thermo


In [ ]:
def calculate_sonde_metrics(drop_final_name, radio_final_name, skip_surface_sondes, lowmax_hght):
    
    """ Calculate mean-layer metrics for dropsondes and radiosondes from a given CAMP2Ex research flight using SHARPpy
    
    PARAMETERS
    ----------
    drop_final_name : the filepath (string) of the desired research flight's QCed dropsonde CSV file, which
                      was created using CAMP2Ex_dropsonde.ipynb

    radio_final_name : the filepath (string) of the desired research flight's QCed radiosonde CSV file, which
                       was created using CAMP2Ex_radiosonde.ipynb

    skip_surface_sondes : a list of sonde datetimes (strings, YYY-MM-DD HH:MM:SS) for which to use the next closest near-surface data
                          (usually 2nd closest near-surface data), rather than the closest near-surface data, for PBL RH and PBL shear
                          calculations. This is done to avoid PBL RH and shear not being calculated for sondes that have good PBL data
                          for all but the closest near-surface point.
    
    lowmax_hght : the height level (in meters) to serve as the upper- and deep-layer cap
    
    
    RETURNS
    ----------
    No return variables, but the function adds the calculated sonde metrics from the desired research 
    flight to Sonde_Metric_Calculations_CAMP2Ex_NEW.csv, located in the directory of this script
    
    """    
    
    drop_csv = pd.read_csv(drop_final_name)
    if os.path.isfile(radio_final_name):
        radio_csv = pd.read_csv(radio_final_name)
        sonde_csv = pd.concat([drop_csv, radio_csv], ignore_index = True)  #concatenates fields with same header
    else:
        sonde_csv = drop_csv.copy()
    sonde_times = sonde_csv['Time [UTC]'].unique()                         #sorted() = goes through files in alphabetical order

    for time in sonde_times:

        print ('Checking sonde at:', time)
        
        rel_data = sonde_csv[sonde_csv['Time [UTC]'] == time].copy()
        rel_data2 = rel_data.iloc[::-1]  #reverses the dataframe (row-based) to go from surface to upper-level

        if time in skip_surface_sondes:   #surface moisture or wind data is missing, which causes PBL RH or shear to not be calculated
            if time == '2019-09-15 22:50:42' or time == '2019-10-05 04:27:46':
                pres = rel_data2['Pressure [mb]'][2:]
                hght = rel_data2['Height [m]'][2:]
                tmpc = rel_data2['Temperature [C]'][2:]
                dwpc = rel_data2['Dew Point [C]'][2:]
                wspd = 1.94384449 * rel_data2['Wind Speed [m/s]'][2:]  #converts m/s to knots (also in SHARPpy sharptab.utils script)
                wdir = rel_data2['Wind Direction [deg]'][2:]
            else:
                pres = rel_data2['Pressure [mb]'][1:]
                hght = rel_data2['Height [m]'][1:]
                tmpc = rel_data2['Temperature [C]'][1:]
                dwpc = rel_data2['Dew Point [C]'][1:]
                wspd = 1.94384449 * rel_data2['Wind Speed [m/s]'][1:]  #converts m/s to knots (also in SHARPpy sharptab.utils script)
                wdir = rel_data2['Wind Direction [deg]'][1:]
        else:
            pres = rel_data2['Pressure [mb]']
            hght = rel_data2['Height [m]']
            tmpc = rel_data2['Temperature [C]']
            dwpc = rel_data2['Dew Point [C]']
            wspd = 1.94384449 * rel_data2['Wind Speed [m/s]']  #converts m/s to knots (also in SHARPpy sharptab.utils script)
            wdir = rel_data2['Wind Direction [deg]']

        list_pres = pres.tolist()   
        list_hght = hght.tolist()
        list_wspd = wspd.tolist()
        list_wdir = wdir.tolist()

        try:
            prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=True)
        except:
            prof = profile.create_profile(profile='default', pres=pres, hght=hght, tmpc=tmpc, dwpc=dwpc, wspd=wspd, wdir=wdir, missing=-9999, strictQC=False)    

        sfc_p = prof.pres.data[prof.sfc]
        sfc_hght = prof.hght.data[prof.sfc]

        freeze_lev = params.temp_lvl(prof, temp = 0)
        #print ('Profile Pressure Value at 0 C: %.1f' % freeze_lev)
        m10c_lev = params.temp_lvl(prof, temp = -10)
        #print ('Profile Pressure Value at -10 C: %.1f' % m10c_lev)
        m20c_lev = params.temp_lvl(prof, temp = -20)
        #print ('Profile Pressure Value at -20 C: %.1f' % m20c_lev)
        m30c_lev = params.temp_lvl(prof, temp = -30)
        #print ('Profile Pressure Value at -30 C: %.1f' % m30c_lev)

        # Calculate PBL Top using the virtual potential temperature method (see Ajda's work and params.pbl_top code)
        pbl_top_pres = params.pbl_top(prof)
        #print ('PBL Top: %.1f mb' % pbl_top_pres) # mb

        PBL_bot_index = 0  #i.e. list_pres.index(sfc_p)
        PBL_top_index = list_pres.index(pbl_top_pres)
        pbl_top_hght = prof.hght.data[PBL_top_index]

        mid_bot_index = PBL_top_index + 1

        # Find the pressure/height value that corresponds closest to the lowest max sonde height without going over
        x = 0
        upper_lvl_hcap = -999.0  #just in case the dropsonde data doesn't reach the lowmax_hght when falling, this will alert you of the dropsonde
        upper_lvl_pcap = -999.0  #just in case the dropsonde data doesn't reach the lowmax_hght when falling, this will alert you of the dropsonde
        while prof.hght.data[x] <= lowmax_hght:
            upper_lvl_pcap = prof.pres.data[x]
            upper_lvl_hcap = prof.hght.data[x]
            if x == (len(prof.hght.data) - 1):       #stops the loop if the highest (lowest) height (pressure) in the profile was just reached/assigned
                break
            else:
                x += 1
        # x = np.argmin(abs(prof.hght.data - lowmax_hght))
        # upper_lvl_pcap = prof.pres.data[x]
        # upper_lvl_hcap = prof.hght.data[x]
        if upper_lvl_hcap < (lowmax_hght - 100):  #i.e., if the sonde data doesn't reach close to the lowmax_hght (i.e., lowmax_hght - 100m)
            print ('WARNING: Sonde at ', time, ' did not reach close (i.e., within 100m) to the lowmax_hght.')
            bad_upper_lvl_cap = True
            upper_top_index = list_pres.index(prof.pres.data[-1])  #sonde minimum pressure
            #continue
        else:
            bad_upper_lvl_cap = False
            upper_top_index = list_pres.index(upper_lvl_pcap)
        
        #Since freeze_lev isn't always a literal pressure entry in the profile, need to find the
            #lowest pressure in the mid layer (i.e., the pressure level closest to the freezing level without going over)
        z = 0
        freeze_lev_hght = -999.0  #just in case the dropsonde data doesn't reach the freezing level when falling, this will alert you of the dropsonde
        mid_top_index = -999.0
        upper_bot_index = -999.0
        while prof.pres.data[z] >= freeze_lev:  #if isinstance(freeze_lev, np.ma.core.MaskedConstant) (i.e., if freeze_lev is NaN), then this condition will always be False
            mid_top_index = z + 0   #(+ 0 is to make sure to avoid weird Python shallow copy, though probably not necessary)
            freeze_lev_hght = prof.hght.data[mid_top_index]
            z += 1
            upper_bot_index = z + 0  #i.e. mid_top_index + 1 (+ 0 is to make sure to avoid weird Python shallow copy, though probably not necessary)
        #z = np.argmin(abs(prof.pres.data - freeze_lev))
        #mid_top_index = prof.pres.data[z]
        #upper_bot_index = prof.pres.data[z+1]
        if freeze_lev_hght < 0:  #i.e., if the sonde data doesn't reach the freezing level (i.e., if isinstance(freeze_lev, np.ma.core.MaskedConstant) (i.e., if freeze_lev is NaN))
            bad_freeze_lev = True
            print ('WARNING: Sonde at ', time, ' did not reach the freezing level.')
            freeze_lev_hght = np.nan
            mid_top_index = list_pres.index(prof.pres.data[-1])    #sonde minimum pressure (won't be used since bad_freeze_lev = True)
            upper_bot_index = list_pres.index(prof.pres.data[-1])  #sonde minimum pressure (won't be used since bad_freeze_lev = True)
            #continue
        else:
            bad_freeze_lev = False

        print ('Working on metrics for sonde at:', time, '\n')
        # print ('Sonde max height: %.1f m' % prof.hght.data[-1])
        # print ('Surface Pressure: %.1f mb' % sfc_p)
        # print ('Upper-level Pressure Cap: %.1f mb' % upper_lvl_pcap)
        
        drop_metrics = [''.join(time[:10].split('-')), time[11:].replace(':',''), '', '', '', '', '', '', '', '', '',
                       '', '', '', '', '', '', '', '', '', '', '', '', '', 'Western Pacific', '', '', '', '', '', '']
        
        #mean-layer RH calculations based off of SHARPpy pressure threshold values (i.e. sfc - PBL Top, PBL Top - freezing level, freezing level - upper_lvl_pcap)
        PBL_RH_average = rel_data2[rel_data2['Pressure [mb]'] >= pbl_top_pres]['Relative Humidity [%]'].mean(skipna = True)
        
        if not bad_freeze_lev:
            mid_RH_average = rel_data2[(rel_data2['Pressure [mb]'] < pbl_top_pres) & (rel_data2['Pressure [mb]'] >= freeze_lev)]['Relative Humidity [%]'].mean(skipna = True)
        else:
            mid_RH_average = np.nan
        if not bad_upper_lvl_cap:
            upper_RH_average = rel_data2[(rel_data2['Pressure [mb]'] < freeze_lev) & (rel_data2['Pressure [mb]'] >= upper_lvl_pcap)]['Relative Humidity [%]'].mean(skipna = True)
            deep_RH_average = rel_data2[rel_data2['Pressure [mb]'] >= upper_lvl_pcap]['Relative Humidity [%]'].mean(skipna = True)
        else:
            upper_RH_average = np.nan
            deep_RH_average = np.nan

        # print ('PBL RH %:  ', np.round(PBL_RH_average,1))
        # print ('Mid Layer RH %:  ', np.round(mid_RH_average,1))  
        # print ('Upper Layer RH %:  ', np.round(upper_RH_average,1))   
        # print ('Deep Layer RH %:  ', np.round(deep_RH_average,1))

        #print ('Paste the following into Sonde_Metric_Calculations.csv:')
        #print ('')
        
        #Metrics independent of parcel type
        drop_metrics.append(np.round(sfc_p, 1))            
        drop_metrics.append(np.round(sfc_hght, 1))
        drop_metrics.append(np.round(pbl_top_pres, 1))
        drop_metrics.append(np.round(pbl_top_hght, 1))
        drop_metrics.append(np.round(freeze_lev, 1))
        drop_metrics.append(np.round(freeze_lev_hght, 1))
        drop_metrics.append(np.round(upper_lvl_pcap, 1))
        drop_metrics.append(np.round(upper_lvl_hcap, 1))
        drop_metrics.append(np.round(m10c_lev, 1))
        drop_metrics.append(np.round(m20c_lev, 1))
        drop_metrics.append(np.round(m30c_lev, 1))
        drop_metrics.append(np.round(PBL_RH_average, 1))
        drop_metrics.append(np.round(mid_RH_average, 1))
        drop_metrics.append(np.round(upper_RH_average, 1))   
        drop_metrics.append(np.round(deep_RH_average, 1))
        

        if (not bad_upper_lvl_cap) and (not bad_freeze_lev): #if the sonde goes up to the freezing level and lowmax_hght
            #mupcl = params.parcelx(prof, flag=3, exact = True) # Most-Unstable Parcel
            #mlpcl = params.parcelx(prof, flag=4, exact = True) # 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            Dmupcl = params.parcelx(prof, flag=3, ptop = upper_lvl_pcap, exact = True) # Deep Layer Most-Unstable Parcel
            Bmupcl = params.parcelx(prof, flag=3, ptop = freeze_lev, exact = True) # Below Freezing Level Most-Unstable Parcel
            Dmlpcl = params.parcelx(prof, flag=4, ptop = upper_lvl_pcap, exact = True) # Deep layer 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            Bmlpcl = params.parcelx(prof, flag=4, ptop = freeze_lev, exact = True) # Below Freezing Level 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)    

            below_mucape = Bmupcl.bplus                        #Below Freezing Level MUCAPE
            deep_mucape = Dmupcl.bplus                         #Deep Layer MUCAPE
            above_mucape = deep_mucape - below_mucape          #Above Freezing Level MUCAPE
            below_mlcape = Bmlpcl.bplus                        #Below Freezing Level MLCAPE
            deep_mlcape = Dmlpcl.bplus                         #Deep Layer MLCAPE
            above_mlcape = deep_mlcape - below_mlcape          #Above Freezing Level MLCAPE

            drop_metrics.append(np.round(below_mucape, 1))     #MU metrics start here     
            drop_metrics.append(np.round(above_mucape, 1))
            drop_metrics.append(np.round(deep_mucape, 1))
            drop_metrics.append(np.round(Dmupcl.bminus, 1))
            drop_metrics.append(np.round(Dmupcl.lclpres, 1))
            drop_metrics.append(np.round(Dmupcl.elpres, 1))
            drop_metrics.append(np.round(Dmupcl.limax, 2))
            drop_metrics.append(np.round(Dmupcl.limaxpres, 1))
            drop_metrics.append('')                            #MU Max LI Layer (inputted manually)
            drop_metrics.append(np.round(below_mlcape, 1))     #ML metrics start here
            drop_metrics.append(np.round(above_mlcape, 1))
            drop_metrics.append(np.round(deep_mlcape, 1))
            drop_metrics.append(np.round(Dmlpcl.bminus, 1))
            drop_metrics.append(np.round(Dmlpcl.lclpres, 1))
            drop_metrics.append(np.round(Dmlpcl.elpres, 1))
            drop_metrics.append(np.round(Dmlpcl.limax, 2))
            drop_metrics.append(np.round(Dmlpcl.limaxpres, 1))
            drop_metrics.append('')                            #ML Max LI Layer (inputted manually)
            
        elif not bad_freeze_lev: #if the sonde just goes up to the freezing level
            #mupcl = params.parcelx(prof, flag=3, exact = True) # Most-Unstable Parcel
            #mlpcl = params.parcelx(prof, flag=4, exact = True) # 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            #Dmupcl = params.parcelx(prof, flag=3, ptop = upper_lvl_pcap, exact = True) # Deep Layer Most-Unstable Parcel
            Bmupcl = params.parcelx(prof, flag=3, ptop = freeze_lev, exact = True) # Below Freezing Level Most-Unstable Parcel
            #Dmlpcl = params.parcelx(prof, flag=4, ptop = upper_lvl_pcap, exact = True) # Deep layer 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            Bmlpcl = params.parcelx(prof, flag=4, ptop = freeze_lev, exact = True) # Below Freezing Level 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)   

            below_mucape = Bmupcl.bplus                        #Below Freezing Level MUCAPE
            #deep_mucape = Dmupcl.bplus                        #Deep Layer MUCAPE
            #above_mucape = deep_mucape - below_mucape         #Above Freezing Level MUCAPE
            below_mlcape = Bmlpcl.bplus                        #Below Freezing Level MLCAPE
            #deep_mlcape = Dmlpcl.bplus                        #Deep Layer MLCAPE
            #above_mlcape = deep_mlcape - below_mlcape         #Above Freezing Level MLCAPE

            drop_metrics.append(np.round(below_mucape, 1))     #MU metrics start here     
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.round(Bmupcl.bminus, 1))
            drop_metrics.append(np.round(Bmupcl.lclpres, 1))
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append('')                            #MU Max LI Layer (inputted manually)
            drop_metrics.append(np.round(below_mlcape, 1))     #ML metrics start here
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.round(Bmlpcl.bminus, 1))
            drop_metrics.append(np.round(Bmlpcl.lclpres, 1))
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append('')                            #ML Max LI Layer (inputted manually)
            
        else: #if the sonde does not go up to the freezing level nor lowmax_hght
            # mupcl = params.parcelx(prof, flag=3, exact = True) # Most-Unstable Parcel
            # mlpcl = params.parcelx(prof, flag=4, exact = True) # 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            # Dmupcl = params.parcelx(prof, flag=3, ptop = upper_lvl_pcap, exact = True) # Deep Layer Most-Unstable Parcel
            # Bmupcl = params.parcelx(prof, flag=3, ptop = freeze_lev, exact = True) # Below Freezing Level Most-Unstable Parcel
            # Dmlpcl = params.parcelx(prof, flag=4, ptop = upper_lvl_pcap, exact = True) # Deep layer 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            # Bmlpcl = params.parcelx(prof, flag=4, ptop = freeze_lev, exact = True) # Below Freezing Level 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
            
            # below_mucape = np.nan                #Below Freezing Level MUCAPE
            # deep_mucape = np.nan                 #Deep Layer MUCAPE
            # above_mucape = np.nan                #Above Freezing Level MUCAPE
            # below_mlcape = np.nan                #Below Freezing Level MLCAPE
            # deep_mlcape = np.nan                 #Deep Layer MLCAPE
            # above_mlcape = np.nan                #Above Freezing Level MLCAPE

            drop_metrics.append(np.nan)            #MU metrics start here     
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append('')                #MU Max LI Layer (inputted manually)
            drop_metrics.append(np.nan)            #ML metrics start here
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append(np.nan)
            drop_metrics.append('')                #ML Max LI Layer (inputted manually)
            
        # try:
        #     #mupcl = params.parcelx(prof, flag=3, exact = True) # Most-Unstable Parcel
        #     #mlpcl = params.parcelx(prof, flag=4, exact = True) # 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
        #     Dmupcl = params.parcelx(prof, flag=3, ptop = upper_lvl_pcap, exact = True) # Deep Layer Most-Unstable Parcel
        #     Bmupcl = params.parcelx(prof, flag=3, ptop = freeze_lev, exact = True) # Below Freezing Level Most-Unstable Parcel
        #     Dmlpcl = params.parcelx(prof, flag=4, ptop = upper_lvl_pcap, exact = True) # Deep layer 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)
        #     Bmlpcl = params.parcelx(prof, flag=4, ptop = freeze_lev, exact = True) # Below Freezing Level 100 mb Mean Layer Parcel (from sfc. to sfc. - 100mb)    

        #     below_mucape = Bmupcl.bplus                #Below Freezing Level MUCAPE
        #     deep_mucape = Dmupcl.bplus                 #Deep Layer MUCAPE
        #     above_mucape = deep_mucape - below_mucape  #Above Freezing Level MUCAPE
        #     below_mlcape = Bmlpcl.bplus                #Below Freezing Level MLCAPE
        #     deep_mlcape = Dmlpcl.bplus                 #Deep Layer MLCAPE
        #     above_mlcape = deep_mlcape - below_mlcape  #Above Freezing Level MLCAPE
        #     no_moisture_data = False
        # except:  #no moisture data for the sonde
        #     no_moisture_data = True

        # if no_moisture_data:            #if no thermodynamic instability metrics available     
        #     drop_metrics.append('')     #MU metrics start here    
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')     #MU Max LI Layer (inputted manually if you want, though not used in analysis)
        #     drop_metrics.append('')     #ML metrics start here
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')
        #     drop_metrics.append('')     #ML Max LI Layer (inputted manually if you want, though not used in analysis)
        # else:    
        #     drop_metrics.append(np.round(below_mucape, 1))     #MU metrics start here     
        #     drop_metrics.append(np.round(above_mucape, 1))
        #     drop_metrics.append(np.round(deep_mucape, 1))
        #     drop_metrics.append(np.round(Dmupcl.bminus, 1))
        #     drop_metrics.append(np.round(Dmupcl.lclpres, 1))
        #     drop_metrics.append(np.round(Dmupcl.elpres, 1))
        #     drop_metrics.append(np.round(Dmupcl.limax, 2))
        #     drop_metrics.append(np.round(Dmupcl.limaxpres, 1))
        #     drop_metrics.append('')                            #MU Max LI Layer (inputted manually)
        #     drop_metrics.append(np.round(below_mlcape, 1))     #ML metrics start here
        #     drop_metrics.append(np.round(above_mlcape, 1))
        #     drop_metrics.append(np.round(deep_mlcape, 1))
        #     drop_metrics.append(np.round(Dmlpcl.bminus, 1))
        #     drop_metrics.append(np.round(Dmlpcl.lclpres, 1))
        #     drop_metrics.append(np.round(Dmlpcl.elpres, 1))
        #     drop_metrics.append(np.round(Dmlpcl.limax, 2))
        #     drop_metrics.append(np.round(Dmlpcl.limaxpres, 1))
        #     drop_metrics.append('')                            #ML Max LI Layer (inputted manually)

        #SHARPpy direct shear calculation method
        if time == '2019-09-17 03:57:33' or time == '2019-09-07 06:35:58' or time == '2019-10-01 22:14:01':  #these sondes are missing wind data only at the exact pbl_top_pres, so use the next highest pressure (next lowest height) for PBL shear calculations instead
            PBL_shear = winds.wind_shear(prof, pbot = sfc_p, ptop = prof.pres.data[PBL_top_index - 1])
        else:
            PBL_shear = winds.wind_shear(prof, pbot = sfc_p, ptop = pbl_top_pres)
        PBL_dir, PBL_speed = utils.comp2vec(PBL_shear[0], PBL_shear[1])

        if not bad_freeze_lev:
            mid_shear = winds.wind_shear(prof, pbot = list_pres[mid_bot_index], ptop = list_pres[mid_top_index])
            mid_dir, mid_speed = utils.comp2vec(mid_shear[0], mid_shear[1])
        else:
            mid_dir = np.nan
            mid_speed = np.nan

        if not bad_upper_lvl_cap:
            upper_shear = winds.wind_shear(prof, pbot = list_pres[upper_bot_index], ptop = upper_lvl_pcap)
            upper_dir, upper_speed = utils.comp2vec(upper_shear[0], upper_shear[1])
            deep_shear = winds.wind_shear(prof, pbot = sfc_p, ptop = upper_lvl_pcap)
            deep_dir, deep_speed = utils.comp2vec(deep_shear[0], deep_shear[1])
        else:
            upper_dir = np.nan
            upper_speed = np.nan
            deep_dir = np.nan
            deep_speed = np.nan

        drop_metrics.append(np.round(PBL_speed, 2))
        drop_metrics.append(np.round(PBL_dir, 1))
        drop_metrics.append(np.round(mid_speed, 2))
        drop_metrics.append(np.round(mid_dir, 1))
        drop_metrics.append(np.round(upper_speed, 2))
        drop_metrics.append(np.round(upper_dir, 1))
        drop_metrics.append(np.round(deep_speed, 2))
        drop_metrics.append(np.round(deep_dir, 1))  

        #relevant theta-gradient mean-layer RH values (for PBL and mid-layer) (not really relevant anymore for your research)
        drop_metrics.append(np.nan)               #10mb Interval Theta Gradient PBL Top [mb], if calculated (see download_files.py)
        drop_metrics.append(np.nan)               #10mb Interval Theta Gradient PBL RH [%], if calculated (see download_files.py)
        drop_metrics.append(np.nan)               #10mb Interval Theta Gradient Mid Layer RH [%], if calculated (see download_files.py)
        drop_metrics.append(prof.hght.data[-1])   #sonde max height for records

        #500m Bottom Cap Deep Layer Shear
        index500 = np.argmin(abs(prof.hght.data - 500))
        p500 = prof.pres.data[index500]
        h500 = prof.hght.data[index500]

        if not bad_upper_lvl_cap:
            deep_500m_shear = winds.wind_shear(prof, pbot = p500, ptop = upper_lvl_pcap)
            deep_500m_dir, deep_500m_speed = utils.comp2vec(deep_500m_shear[0], deep_500m_shear[1])
        else:
            deep_500m_dir = np.nan
            deep_500m_speed = np.nan

        drop_metrics.append(np.round(h500, 1))    #sonde height closest to 500m (used for 500m - upper_lvl_cap deep shear calculation)
        drop_metrics.append(np.round(deep_500m_speed, 2))
        drop_metrics.append(np.round(deep_500m_dir, 1))
        print ('')
                    
        df_add = pd.DataFrame(data = [drop_metrics], columns = pd.read_csv('Sonde_Metric_Calculations_CAMP2Ex.csv').columns)
        df_add = df_add.replace(to_replace = np.nan, value = '')
        df_add = df_add.replace(to_replace = '--', value = '')   #this does not get rid of np.ma.core.MaskedConstant values
                                                                 #(which show up as '--' in the CSV), but setting to_replace as
                                                                 #np.ma.core.MaskedConstant throws an error, so just manually replace '--'
                                                                 #with '' in the CSV using find/replace feature
        
        #add the sonde metrics to Sonde_Metric_Calculations_CAMP2Ex_NEW.csv
        if os.path.isfile('Sonde_Metric_Calculations_CAMP2Ex_NEW.csv'):
            df_drop_original = pd.read_csv('Sonde_Metric_Calculations_CAMP2Ex_NEW.csv')
            df_drop_full = pd.concat([df_drop_original, df_add], ignore_index = True)  #concatenates fields with same header
            df_drop_full.to_csv('Sonde_Metric_Calculations_CAMP2Ex_NEW.csv', index = False)
        else: #first sonde of the first research flight, so need to create the new CSV file
            df_add.to_csv('Sonde_Metric_Calculations_CAMP2Ex_NEW.csv', index = False)
            

In [ ]:
#for each CAMP2Ex research flight, open the research flight's QCed dropsonde and radiosonde CSV files and
    #calculate metrics for each dropsonde/radiosonde. Metrics will be added into Sonde_Metric_Calculations_CAMP2Ex_NEW.csv

#surface moisture or wind data is missing, which causes PBL RH or shear to initially not be calculated for these sondes.
    #This list tells the calculate_sonde_metrics() function to just use the next (usually 2nd) closest near-surface data instead,
    #so that PBL RH and shear are calculated
skip_surface_sondes = ['2019-09-04 03:49:03', '2019-08-29 23:46:15', '2019-09-07 06:35:58', '2019-09-09 00:59:25',
                       '2019-10-01 22:14:01', '2019-10-04 01:03:23', '2019-09-07 03:05:51', '2019-09-15 22:50:42',
                       '2019-09-17 02:47:11', '2019-10-05 04:27:46', '2019-10-05 05:58:39', '2019-10-05 06:08:21']
                       #'2019-09-04 03:49:03' is for moisture, the rest are for wind; and don't forget about
                       #'2019-09-17 03:57:33', '2019-09-07 06:35:58', and '2019-10-01 22:14:01', which are hardcoded in the function
                       #as its wind data at the exact PBL top height is missing

for x in sorted(os.listdir()):
    if os.path.isdir(x) and x[0:4] == '2019':
        day_folder = os.path.join(os.getcwd(), x)   
        drop_final_name = os.path.join(day_folder, 'final_dropsonde_' + x + '.csv')
        radio_final_name = os.path.join(day_folder, 'final_radiosonde_' + x + '.csv')
        calculate_sonde_metrics(drop_final_name, radio_final_name, skip_surface_sondes, lowmax_hght = 7622.5)
